# 05 · Write the results report

**Goal:** turn the database into a plain-language, **strictly descriptive** results file,
`05_Documentation/4_Event_Results.md`. It reports what moved, by how much, how unusual the move was, and what else was
going on. It never says an event *caused* a move.

> **Series:** 01 Ingest → 02 Align & transform → 03 Event windows → 04 Database → 05 Results report → 06 Analysis & charts. Run them in order (or run `python run_pipeline.py` from the project folder). Paths and settings live in `config.py`.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

import sqlite3
import pandas as pd
from config import DB_PATH, DOCS, WINDOWS

WORDING = {
    "INR weakened": "USD/INR rose (rupee weaker)",
    "INR strengthened": "USD/INR fell (rupee stronger)",
}

In [2]:
def fmt(x, unit="%", nd=2):
    return "–" if pd.isna(x) else f"{x:+.{nd}f}{unit}"


def write_results_report():
    con = sqlite3.connect(DB_PATH)
    ev = pd.read_sql("SELECT * FROM dim_event ORDER BY event_id", con)
    ew = pd.read_sql("SELECT * FROM fact_event_window", con)
    ctx = pd.read_sql("SELECT * FROM fact_event_window_context", con)
    cor = pd.read_sql("SELECT * FROM fact_event_window_corr", con)
    dq = pd.read_sql("SELECT metric, value FROM dq_summary", con).set_index("metric")["value"]
    wids = [w[0] for w in WINDOWS]
    L = []
    L.append("# Event-window results (auto-generated, descriptive only)\n")
    L.append("> Generated by `03_Notebooks/05_Write_Results_Report.ipynb` from `02_Database/geo_inr_analytics.db`. Every number below is "
             "a **temporal association**: what USD/INR and other markets did around an event date. "
             "None of it shows that the event caused the move. See `05_Documentation/1_Methodology.md` §8.\n")
    L.append("**Sign convention:** USD/INR change > 0 means the rupee **weakened**. "
             "Windows: pre = t(−k)→t(−1); event day = t(−1)→t(0); post = t(−1)→t(+k), in trading days.\n")
    L.append(f"**Primary FX basis:** RBI/FBIL reference rate (data to {int(dq['rbi_last_date'])}). "
             "The FRED (Fed H.10, New York noon) basis is a cross-check.\n")

    # summary matrix
    for basis in ["RBI", "FRED"]:
        m = ew[(ew.basis == basis) & (ew.series_id == "USDINR")].copy()
        m["cell"] = [fmt(c) + ("*" if s == "partial" else "") if s in ("complete", "partial") else "n/a"
                     for c, s in zip(m.change, m.status)]
        piv = m.pivot(index="event_id", columns="window_id", values="cell").reindex(columns=wids)
        L.append(f"\n## USD/INR % change by window — {basis} basis\n")
        L.append("| Event | " + " | ".join(wids) + " |")
        L.append("|---" * (len(wids) + 1) + "|")
        for eid, r in piv.iterrows():
            L.append(f"| {eid} | " + " | ".join(str(x) if pd.notna(x) else "n/a" for x in r) + " |")
        L.append("\n`*` = partial window (calendar ends before t(+k)); `n/a` = not covered.\n")

    # per-event detail
    L.append("\n## Per-event detail (RBI basis)\n")
    for _, e in ev.iterrows():
        basis = "RBI"
        sub = ew[(ew.basis == basis) & (ew.event_id == e.event_id)]
        L.append(f"\n### {e.event_id} — {e.event_name}\n")
        L.append(f"- **Dates:** start {e.event_start_date}"
                 + (f", end {e.event_end_date}" if isinstance(e.event_end_date, str) and e.event_end_date else ", no end date")
                 + f"; anchor (public-information date) {e.anchor_date}.")
        L.append(f"- **Source:** {e.source_org}, *{e.source_title}* ({e.source_pub_date}) — {e.source_url}")
        rows = []
        for w in ["EVENT", "POST_7", "POST_30", "POST_90"]:
            u = sub[(sub.window_id == w) & (sub.series_id == "USDINR")]
            if u.empty or u.iloc[0].status not in ("complete", "partial"):
                continue
            u = u.iloc[0]
            others = {s: sub[(sub.window_id == w) & (sub.series_id == s)]["change"].squeeze()
                      for s in ["BRENT", "GOLD_USD", "DXY_BROAD", "NIFTY50", "VIX"]}
            rows.append(f"| {w}{' (partial)' if u.status == 'partial' else ''} | {fmt(u.change)} | "
                        f"{fmt(u.usdinr_minus_dxy_pp, ' pp')} | "
                        f"{'–' if pd.isna(u.percentile) else f'{u.percentile:.1f}'} | "
                        + " | ".join(fmt(others[s]) if not isinstance(others[s], pd.Series) else "–"
                                     for s in others) + " |")
        if rows:
            L.append("\n| Window | USD/INR | USD/INR − broad USD | Percentile vs prior 3y | Brent | Gold | Broad USD | Nifty 50 | VIX |")
            L.append("|---|---|---|---|---|---|---|---|---|")
            L += rows
        c30 = cor[(cor.basis == basis) & (cor.event_id == e.event_id) & (cor.window_id == "POST_30")]
        if not c30.empty:
            L.append("\n- **Daily-return correlation with USD/INR, t(−1)→t(+30):** " + ", ".join(
                f"{r.series_id} {r.corr:+.2f}" for r in c30.itertuples()) + " (n≈30 — very noisy).")
        cx = ctx[(ctx.basis == basis) & (ctx.event_id == e.event_id) & (ctx.window_id == "POST_90")]
        if not cx.empty:
            cx = cx.iloc[0]
            L.append(f"- **Inside the +90 window:** RBI repo changes: {cx.rbi_repo_changes or 'none'}; "
                     f"FOMC target changes: {cx.fomc_target_changes or 'none'}; "
                     f"other events: {cx.overlapping_events or 'none'}; "
                     f"RBI net FX purchase(+)/sale(−) over {cx.rbi_fx_months}: "
                     + ("n/a" if pd.isna(cx.rbi_net_fx_usd_mn) else f"{cx.rbi_net_fx_usd_mn/1000:+.1f} bn USD") + ".")
        L.append(f"- **Concurrent factors (documented):** {e.known_concurrent_factors}")
    L.append("\n---\nPercentile = share of all same-length USD/INR changes in the ~750 trading days "
             "before the window that were smaller than the event-window change. It is descriptive: "
             "overlapping windows are autocorrelated, so it is **not** a significance test.\n")
    (DOCS / "4_Event_Results.md").write_text("\n".join(L), encoding="utf-8")
    print("wrote 05_Documentation/4_Event_Results.md")

In [3]:
write_results_report()

wrote 05_Documentation/4_Event_Results.md


## Preview

In [4]:
from IPython.display import Markdown
text = (DOCS / "4_Event_Results.md").read_text(encoding="utf-8")
Markdown(text[: text.index("## Per-event detail")])

# Event-window results (auto-generated, descriptive only)

> Generated by `03_Notebooks/05_Write_Results_Report.ipynb` from `02_Database/geo_inr_analytics.db`. Every number below is a **temporal association**: what USD/INR and other markets did around an event date. None of it shows that the event caused the move. See `05_Documentation/1_Methodology.md` §8.

**Sign convention:** USD/INR change > 0 means the rupee **weakened**. Windows: pre = t(−k)→t(−1); event day = t(−1)→t(0); post = t(−1)→t(+k), in trading days.

**Primary FX basis:** RBI/FBIL reference rate (data to 20260929). The FRED (Fed H.10, New York noon) basis is a cross-check.


## USD/INR % change by window — RBI basis

| Event | PRE_30 | PRE_14 | PRE_7 | EVENT | POST_7 | POST_14 | POST_30 | POST_90 |
|---|---|---|---|---|---|---|---|---|
| E01 | +0.88% | -0.39% | -0.08% | -0.34% | -1.58% | -2.21% | -2.19% | -2.81% |
| E02 | -0.56% | -0.00% | -0.85% | +0.16% | +0.21% | +0.67% | +2.00% | +0.74% |
| E03 | +0.20% | +0.33% | +0.01% | +0.41% | +0.05% | -0.07% | -0.47% | +1.48% |
| E04 | +1.54% | -0.27% | -0.88% | +0.44% | +0.32% | -1.42% | -2.95% | -2.94% |
| E05 | -0.64% | +0.88% | +0.21% | +0.49% | -0.59% | -0.14% | +0.06% | +6.02% |
| E06 | +1.32% | +0.61% | +0.85% | -0.35% | -0.62% | -1.87% | -1.79% | -3.32% |
| E07 | +0.93% | -0.31% | -1.37% | +0.85% | +3.03% | +1.70% | +2.00% | +6.42% |
| E08 | +0.83% | +0.22% | +0.02% | +0.02% | +0.02% | +0.03% | +0.12% | -0.42% |
| E09 | +0.61% | +0.55% | +0.04% | +0.06% | -0.02% | +0.13% | -0.11% | +0.69% |
| E10 | -2.61% | -0.77% | -1.75% | +0.40% | -1.01% | +0.57% | +0.81% | +3.44% |
| E11 | +0.57% | +0.75% | -0.49% | +0.66% | +0.67% | -0.03% | +1.16% | +2.96% |
| E12 | +0.83% | +0.61% | +0.32% | +0.55% | +1.54% | +3.22% | +3.17% | +6.11% |
| E13 | +0.11% | +0.99% | +1.02% | -0.09% | +1.10% | +0.51% | +0.46% | +0.75%* |

`*` = partial window (calendar ends before t(+k)); `n/a` = not covered.


## USD/INR % change by window — FRED basis

| Event | PRE_30 | PRE_14 | PRE_7 | EVENT | POST_7 | POST_14 | POST_30 | POST_90 |
|---|---|---|---|---|---|---|---|---|
| E01 | +0.39% | -1.06% | -0.69% | +0.63% | -1.25% | -1.42% | -2.36% | -3.29% |
| E02 | -0.69% | -0.39% | -0.70% | +0.33% | +0.47% | +0.50% | +1.92% | +0.74% |
| E03 | -0.32% | +0.28% | +0.22% | -0.06% | -0.20% | -0.43% | -0.90% | +0.43% |
| E04 | +1.66% | +0.17% | -0.52% | +0.25% | -0.49% | -1.91% | -3.19% | -2.81% |
| E05 | -0.72% | +1.02% | +0.14% | +0.59% | -0.69% | -0.04% | +0.31% | +5.17% |
| E06 | +0.44% | +0.12% | +0.48% | +0.29% | -0.51% | -1.53% | -1.38% | -2.78% |
| E07 | +1.07% | -0.15% | -1.36% | +1.35% | +3.10% | +2.10% | +1.74% | +6.26% |
| E08 | +0.74% | -0.02% | +0.14% | +0.00% | -0.18% | +0.00% | +0.05% | -0.35% |
| E09 | +0.86% | +0.32% | +0.18% | -0.13% | -0.30% | -0.19% | -0.51% | +0.41% |
| E10 | -2.26% | -0.29% | -0.96% | +0.27% | -0.81% | +0.19% | +0.72% | +3.47% |
| E11 | +1.15% | +0.45% | -0.35% | +0.62% | +0.57% | +0.36% | +1.50% | +2.69% |
| E12 | +0.85% | +0.29% | +0.39% | +0.48% | +1.10% | +2.97% | +2.59% | +4.79% |
| E13 | -0.75% | +0.37% | +0.64% | +0.62% | +1.38% | +0.93% | +0.82% | +0.88%* |

`*` = partial window (calendar ends before t(+k)); `n/a` = not covered.




Next: **06 · Analysis & charts**.